In [1]:
import pandas as pd
import numpy as np
import joblib

def run_inference(data_input):
    """
    Fungsi untuk memprediksi cluster nasabah baru dengan alur lengkap.
    """
    try:
        # 1. LOAD SEMUA KOMPONEN (ALAT)
        medians = joblib.load('medians_data.pkl')
        scaler = joblib.load('scaler_model.pkl')
        pca = joblib.load('pca_model.pkl')
        kmeans = joblib.load('kmeans_model.pkl')

        # 2. CONVERT INPUT KE DATAFRAME
        df_new = pd.DataFrame([data_input])

        # 3. STEP 1: HANDLING MISSING VALUE (Median)
        # Mengisi kolom kosong jika ada data yang tidak diisi
        df_filled = df_new.fillna(medians)

        # 4. STEP 2: LOG TRANSFORM (PENTING!)
        # Mengubah angka normal menjadi skala logaritma agar dipahami model
        df_log = np.log(df_filled + 1)

        # 5. STEP 3: SCALING (Standardization)
        # Menggunakan 'scaler' yang sudah di-load
        df_scaled = scaler.transform(df_log)

        # 6. STEP 4: PCA (Reduksi Dimensi)
        # Mengubah 17 kolom menjadi 5 komponen PCA
        df_pca = pca.transform(df_scaled)

        # 7. STEP 5: PREDIKSI CLUSTER
        cluster_result = kmeans.predict(df_pca)
        
        return int(cluster_result[0])

    except Exception as e:
        return f"Terjadi kesalahan: {str(e)}"

# --- AREA PENGUJIAN DATA NASABAH ID 9999 ---
if __name__ == "__main__":
    # Data input mentah (angka normal)
    new_customer = {
        'BALANCE': 2000, 'BALANCE_FREQUENCY': 0.5, 'PURCHASES': 1000,
        'ONEOFF_PURCHASES': 600, 'INSTALLMENTS_PURCHASES': 800, 'CASH_ADVANCE': 500,
        'PURCHASES_FREQUENCY': 5, 'ONEOFF_PURCHASES_FREQUENCY': 0.5,
        'PURCHASES_INSTALLMENTS_FREQUENCY': 0.7, 'CASH_ADVANCE_FREQUENCY': 0.3,
        'CASH_ADVANCE_TRX': 3, 'PURCHASES_TRX': 20, 'CREDIT_LIMIT': 5000,
        'PAYMENTS': 800, 'MINIMUM_PAYMENTS': 500, 'PRC_FULL_PAYMENT': 0.2, 'TENURE': 12
    }

    # Jalankan Fungsi Inference
    hasil_cluster = run_inference(new_customer)
    
    # Mapping Nama Cluster agar lebih informatif
    nama_cluster = {
        0: "Conservative (Hemat)",
        1: "Cash Borrowers (Peminjam Tunai)",
        2: "High-Value (Top Shopper)"
    }
    
    print("\n" + "="*45)
    print(f"HASIL PREDIKSI UNTUK NASABAH ID 9999")
    print("-" * 45)
    if isinstance(hasil_cluster, int):
        print(f"Cluster ID : {hasil_cluster}")
        print(f"Kategori   : {nama_cluster.get(hasil_cluster)}")
    else:
        print(hasil_cluster)
    print("="*45 + "\n")



HASIL PREDIKSI UNTUK NASABAH ID 9999
---------------------------------------------
Cluster ID : 2
Kategori   : High-Value (Top Shopper)

